<a href="https://colab.research.google.com/github/rselvak6/opendde-colab/blob/main/visualize_opendde_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenDDE Run Visualizer

Standalone viewer for OpenDDE prediction outputs. Upload a `.zip` of a completed run's `output/` directory (the same file the `opendde-colab` notebook's "Download output" cell produces via `shutil.make_archive`), and this notebook lets you:

- Browse every model's structure + PAE matrix (py2Dmol)
- View an AlphaFold3-server-style ribbon cartoon with an interactive ipSAE/ipTM residue matrix
- Compute ipSAE and rank all models by decreasing ipSAE

Re-run the upload cell any time to add another run's `.zip` - models from every run you've uploaded this session stay available together, labeled by which run they came from, so you can keep using this same notebook for future modeling runs.

**Expected zip structure** (what you get zipping an OpenDDE `output/` directory):
```
seed_<seed>/predictions/<name>_sample_<rank>.cif
seed_<seed>/predictions/<name>_summary_confidence_sample_<rank>.json
seed_<seed>/predictions/<name>_full_data_sample_<rank>.json   (needs --need_atom_confidence true)
```

In [ ]:
!pip install -q py3Dmol py2Dmol ipywidgets pandas

In [ ]:
#@title Upload a run's output.zip (re-run to add more runs)
import os
import zipfile

from google.colab import files

RUNS_DIR = "./runs"
os.makedirs(RUNS_DIR, exist_ok=True)

uploaded = files.upload()
for filename, content in uploaded.items():
    if not filename.lower().endswith(".zip"):
        print(f"Skipping {filename}: not a .zip file")
        continue
    run_name = os.path.splitext(filename)[0]
    run_dir = os.path.join(RUNS_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)
    with open(filename, "wb") as f:
        f.write(content)
    with zipfile.ZipFile(filename) as zf:
        zf.extractall(run_dir)
    os.remove(filename)
    print(f"Extracted {filename} -> {run_dir}")

print()
print("Runs available:", sorted(os.listdir(RUNS_DIR)) if os.path.isdir(RUNS_DIR) else [])

In [ ]:
#@title (Optional) clear all uploaded runs to start fresh
import shutil

if os.path.isdir(RUNS_DIR):
    shutil.rmtree(RUNS_DIR)
os.makedirs(RUNS_DIR, exist_ok=True)
print("Cleared", RUNS_DIR)

In [ ]:
#@title Shared helpers: locate each model's .cif plus its matching per-sample
# JSON sidecars, following OpenDDE's dumper.py naming convention:
#   seed_{seed}/predictions/{name}_sample_{rank}.cif
#   seed_{seed}/predictions/{name}_summary_confidence_sample_{rank}.json   (always written)
#   seed_{seed}/predictions/{name}_full_data_sample_{rank}.json             (only with --need_atom_confidence true)
import glob
import os
import re

_SAMPLE_RE = re.compile(r"(.+)_sample_(\d+)\.cif$")


def find_all_cifs(root_dir=RUNS_DIR):
    return sorted(glob.glob(os.path.join(root_dir, "**", "*.cif"), recursive=True))


def _sample_stem_rank(cif_path):
    m = _SAMPLE_RE.match(os.path.basename(cif_path))
    return m.groups() if m else (None, None)


def find_full_data_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_full_data_sample_{rank}.json")
    return p if os.path.exists(p) else None


def find_summary_json(cif_path):
    stem, rank = _sample_stem_rank(cif_path)
    if stem is None:
        return None
    p = os.path.join(os.path.dirname(cif_path), f"{stem}_summary_confidence_sample_{rank}.json")
    return p if os.path.exists(p) else None


def model_label(cif_path):
    # "<run>: seed N, sample M" so models from different uploaded runs stay
    # distinguishable when browsing/ranking them together.
    rel = os.path.relpath(cif_path, RUNS_DIR)
    run_name = rel.split(os.sep)[0]
    _, rank = _sample_stem_rank(cif_path)
    seed_match = re.search(r"seed_(\d+)", cif_path)
    seed = seed_match.group(1) if seed_match else "?"
    base = f"seed {seed}, sample {rank}" if rank is not None else os.path.basename(cif_path)
    return f"{run_name}: {base}"


cifs = find_all_cifs()
run_names = sorted(set(model_label(c).split(":")[0] for c in cifs))
print(f"Found {len(cifs)} predicted structure(s) across {len(run_names)} run(s): {run_names}")

In [ ]:
#@title Visualize predicted structures + PAE (py2Dmol)
import json

import numpy as np
import py2Dmol

load_as_frames = True  # @param {type:"boolean"}
viewer_size = 400


def load_pae(cif_path):
    # OpenDDE's full_data.json stores the PAE matrix as "token_pair_pae"
    # (AlphaFold3's equivalent full_data.json key is "pae").
    fd_path = find_full_data_json(cif_path)
    if fd_path is None:
        return None
    with open(fd_path) as f:
        data = json.load(f)
    pae = data.get("token_pair_pae")
    return np.asarray(pae, dtype=float) if pae is not None else None


if not cifs:
    print("No .cif files found under ./runs yet - upload a run's output.zip first.")
else:
    viewer = py2Dmol.view(size=(viewer_size, viewer_size), pae=True, autoplay=load_as_frames)
    for i, cif in enumerate(cifs, start=1):
        pae = load_pae(cif)
        if pae is None:
            print(f"No PAE data for {cif} (re-run pred with --need_atom_confidence true)")
        name = "models" if load_as_frames else model_label(cif)
        viewer.add_pdb(cif, name=name, paes=pae)
    viewer.show()

In [ ]:
#@title Ribbon (cartoon) visualization with interactive ipSAE/ipTM matrix, like the AlphaFold3 server
import ipywidgets as widgets
from IPython.display import display, HTML
import json
import uuid


def _calc_d0(L):
    # Same d0 formula ipsae.py uses (Yang & Skolnick, 2004), for converting
    # PAE into a pTM-style [0,1] confidence value.
    L = float(L)
    if L > 27:
        return max(1.0, 1.24 * (L - 15) ** (1.0 / 3.0) - 1.8)
    return 1.0


def _flatten_scalar(v):
    while isinstance(v, list):
        v = v[0] if v else None
        if v is None:
            return None
    return float(v) if v is not None else None


def get_iptm_ptm(cif_path):
    summary_path = find_summary_json(cif_path)
    if summary_path is None:
        return None, None
    with open(summary_path) as f:
        data = json.load(f)
    iptm = _flatten_scalar(data.get("iptm"))
    ptm = _flatten_scalar(data.get("ptm"))
    return iptm, ptm


def _build_model_html(cif_path):
    with open(cif_path) as f:
        cif_text = f.read()

    iptm, ptm = get_iptm_ptm(cif_path)

    pae = chain_ids = resi_nums = d0 = None
    full_data_path = find_full_data_json(cif_path)
    if full_data_path is not None:
        with open(full_data_path) as f:
            fd = json.load(f)
        pae = fd.get("token_pair_pae")
        token_asym = fd.get("token_asym_id")

        if pae is not None and token_asym is not None:
            n_tokens = len(token_asym)
            # Assign chain letters + within-chain residue numbers from
            # token_asym_id order (first distinct asym seen -> 'A', next -> 'B', ...).
            chain_ids, resi_nums = [], []
            seen_asym, counters = {}, {}
            for asym in token_asym:
                if asym not in seen_asym:
                    seen_asym[asym] = chr(ord("A") + len(seen_asym))
                    counters[asym] = 0
                counters[asym] += 1
                chain_ids.append(seen_asym[asym])
                resi_nums.append(counters[asym])

            d0 = _calc_d0(n_tokens)
        else:
            pae = None

    uid = uuid.uuid4().hex[:8]
    has_matrix = pae is not None

    score_line = ""
    if iptm is not None or ptm is not None:
        parts = []
        if iptm is not None:
            parts.append(f"ipTM: {iptm:.2f}")
        if ptm is not None:
            parts.append(f"pTM: {ptm:.2f}")
        score_line = f'<div style="font-weight:bold; font-size:15px; margin-bottom:8px;">{"&nbsp;&nbsp;&nbsp;".join(parts)}</div>'

    matrix_html = ""
    if has_matrix:
        matrix_html = f"""
        <div>
          <label>Color by:
            <select id="scoremode_{uid}">
              <option value="ipsae" selected>ipSAE</option>
              <option value="iptm">ipTM</option>
            </select>
          </label>
          &nbsp;&nbsp;
          <label><input type="checkbox" id="boundaries_{uid}" checked> Show chain boundaries</label>
          <br/>
          <canvas id="matrix_{uid}" width="490" height="470"
                  style="cursor:crosshair;"></canvas>
          <div id="legend_{uid}" style="font-size:12px; color:#333; margin-top:2px;"></div>
          <div style="font-size:12px; color:#666;">Drag a box on the matrix to highlight
          the corresponding residues on the structure. Double-click the structure to reset.</div>
        </div>
        """

    html = f"""
    {score_line}
    <div style="display:flex; gap:16px; align-items:flex-start;">
      <div id="viewer_{uid}" style="width:480px; height:420px; position:relative;"></div>
      {matrix_html}
    </div>
    <script>
    (function() {{
      function whenReady(cb) {{
        if (window.$3Dmol) {{ cb(); return; }}
        var s = document.createElement('script');
        s.src = 'https://cdn.jsdelivr.net/npm/3dmol@2.5.5/build/3Dmol-min.js';
        s.onload = cb;
        document.head.appendChild(s);
      }}

      whenReady(function() {{
        var cifText = {json.dumps(cif_text)};
        var baseStyle = {{cartoon: {{colorscheme: {{prop: 'b', gradient: 'linear', min: 0, max: 100,
                          colors: ['#FF7D45', '#FFDB13', '#65CBF3', '#0053D6']}}}}}};

        var element = document.getElementById('viewer_{uid}');
        var viewer = $3Dmol.createViewer(element, {{backgroundColor: 'white'}});
        viewer.addModel(cifText, 'cif');
        viewer.setStyle({{}}, baseStyle);
        viewer.zoomTo();
        viewer.render();

        // 3Dmol.js's built-in scroll-zoom (bound internally on a child canvas)
        // uses an asymmetric formula, so zoom-in/zoom-out speed don't match and
        // the direction isn't configurable. Intercept the wheel event here, in
        // the capture phase, and stop it from ever reaching that inner handler,
        // then drive viewer.zoom() ourselves with a symmetric exponential curve:
        // scrolling forward (deltaY < 0) zooms in, backward zooms out, and the
        // same scroll distance in either direction exactly cancels out.
        element.addEventListener('wheel', function(e) {{
          e.preventDefault();
          e.stopPropagation();
          var factor = Math.pow(1.0015, -e.deltaY);
          viewer.zoom(factor, 0);
        }}, {{capture: true, passive: false}});

        var paeMatrix = {json.dumps(pae) if has_matrix else 'null'};
        var chainIds = {json.dumps(chain_ids) if has_matrix else 'null'};
        var resiNums = {json.dumps(resi_nums) if has_matrix else 'null'};
        var d0Global = {json.dumps(d0) if has_matrix else 'null'};
        if (!paeMatrix) return;

        var N = paeMatrix.length;
        var PAE_CUTOFF = 10;
        var leftMargin = 60, topPad = 10, bottomMargin = 50, rightPad = 10;
        var matrixSize = 400;
        var cellSize = matrixSize / N;

        var canvas = document.getElementById('matrix_{uid}');
        var ctx = canvas.getContext('2d');
        var offCanvas = document.createElement('canvas');
        offCanvas.width = N; offCanvas.height = N;
        var offCtx = offCanvas.getContext('2d');

        function iptmColor(v) {{
          v = Math.max(0, Math.min(1, v));
          var c0 = [255, 255, 255], c1 = [0, 51, 153];
          return [Math.round(c0[0] + (c1[0] - c0[0]) * v),
                  Math.round(c0[1] + (c1[1] - c0[1]) * v),
                  Math.round(c0[2] + (c1[2] - c0[2]) * v)];
        }}

        // Same d0 formula as _calc_d0 above (Yang & Skolnick, 2004).
        function calcD0(L) {{
          if (L > 27) return Math.max(1.0, 1.24 * Math.pow(L - 15, 1.0 / 3.0) - 1.8);
          return 1.0;
        }}

        // Per-row, per-target-chain d0 for the primary (d0res-based) ipSAE
        // metric - the same one ipsae.py reports as its plain "ipSAE" column
        // (as opposed to the ipSAE_d0chn/ipSAE_d0dom variants) and the one
        // used to build ipsae_ranking.csv. For row i, each partner chain gets
        // its own d0 based on how many of i's residues in that chain fall
        // under the PAE cutoff. undefined here means zero valid partners.
        var d0resByRow = [];
        for (var ri = 0; ri < N; ri++) {{
          var counts = {{}};
          for (var rj = 0; rj < N; rj++) {{
            if (chainIds[rj] === chainIds[ri]) continue;
            if (paeMatrix[ri][rj] < PAE_CUTOFF) {{
              counts[chainIds[rj]] = (counts[chainIds[rj]] || 0) + 1;
            }}
          }}
          var perChain = {{}};
          for (var ch in counts) perChain[ch] = calcD0(counts[ch]);
          d0resByRow.push(perChain);
        }}

        // Returns {{values, max}} for the given mode. ipSAE values in real
        // predictions are often tiny (sparse/weak interfaces routinely peak
        // well under 0.1), so ipSAE mode auto-scales white->blue to this
        // model's own max value instead of a fixed 0-1 scale, or the
        // interface would just look blank. The legend below the matrix
        // always shows what the darkest blue actually corresponds to.
        function computeValues(mode) {{
          var values = [];
          var max = 0;
          for (var i = 0; i < N; i++) {{
            var row = [];
            for (var j = 0; j < N; j++) {{
              var v;
              if (mode === 'ipsae') {{
                if (chainIds[i] === chainIds[j]) {{
                  v = null;
                }} else {{
                  var rowD0 = d0resByRow[i][chainIds[j]];
                  // ipsae.py returns exactly 0.0 when a row has no residues
                  // under the PAE cutoff in the target chain, rather than
                  // computing ptm_func with some fallback d0.
                  v = (rowD0 === undefined) ? 0.0
                      : 1.0 / (1.0 + Math.pow(paeMatrix[i][j] / rowD0, 2.0));
                }}
              }} else {{
                v = 1.0 / (1.0 + Math.pow(paeMatrix[i][j] / d0Global, 2.0));
              }}
              row.push(v);
              if (v !== null && v > max) max = v;
            }}
            values.push(row);
          }}
          return {{values: values, max: max}};
        }}

        function drawBase(mode) {{
          var computed = computeValues(mode);
          var values = computed.values;
          var max = mode === 'ipsae' ? computed.max : 1.0;
          var imgData = offCtx.createImageData(N, N);
          for (var i = 0; i < N; i++) {{
            for (var j = 0; j < N; j++) {{
              var v = values[i][j];
              var c = (v === null) ? [225, 225, 225] : iptmColor(max > 0 ? v / max : 0);
              var idx = (i * N + j) * 4;
              imgData.data[idx] = c[0]; imgData.data[idx + 1] = c[1];
              imgData.data[idx + 2] = c[2]; imgData.data[idx + 3] = 255;
            }}
          }}
          offCtx.putImageData(imgData, 0, 0);

          var legend = document.getElementById('legend_{uid}');
          if (mode === 'ipsae') {{
            legend.textContent = max > 0
              ? 'White = 0, darkest blue = ' + max.toFixed(3) + ' (this model\'s max ipSAE value; interfaces are often faint, so this is scaled to what\'s actually present, not a fixed 0-1 range).'
              : 'No inter-chain residue pairs fall under the PAE cutoff for this model - ipSAE is 0 everywhere.';
          }} else {{
            legend.textContent = 'White = 0, darkest blue = 1.0 (fixed scale, global d0 = ' + d0Global.toFixed(2) + ').';
          }}
        }}

        function niceTickStep(n, targetTicks) {{
          var raw = n / targetTicks;
          var pow10 = Math.pow(10, Math.floor(Math.log10(raw)));
          var norm = raw / pow10;
          var step = norm < 1.5 ? 1 : norm < 3 ? 2 : norm < 7 ? 5 : 10;
          return Math.max(1, Math.round(step * pow10));
        }}
        var tickStep = niceTickStep(N, 8);

        function drawAxes() {{
          ctx.fillStyle = '#000';
          ctx.strokeStyle = '#999';
          ctx.lineWidth = 1;
          ctx.font = '11px sans-serif';

          // Left axis ("Aligned residue", rows)
          ctx.textAlign = 'right';
          ctx.textBaseline = 'middle';
          for (var i = 0; i < N; i += tickStep) {{
            var y = topPad + (i + 0.5) * cellSize;
            ctx.beginPath();
            ctx.moveTo(leftMargin - 4, y);
            ctx.lineTo(leftMargin, y);
            ctx.stroke();
            ctx.fillText(String(i + 1), leftMargin - 7, y);
          }}
          ctx.save();
          ctx.translate(14, topPad + matrixSize / 2);
          ctx.rotate(-Math.PI / 2);
          ctx.textAlign = 'center';
          ctx.font = 'bold 12px sans-serif';
          ctx.fillText('Aligned residue', 0, 0);
          ctx.restore();

          // Bottom axis ("Scored residue", columns)
          ctx.font = '11px sans-serif';
          ctx.textAlign = 'center';
          ctx.textBaseline = 'top';
          for (var j = 0; j < N; j += tickStep) {{
            var x = leftMargin + (j + 0.5) * cellSize;
            ctx.beginPath();
            ctx.moveTo(x, topPad + matrixSize);
            ctx.lineTo(x, topPad + matrixSize + 4);
            ctx.stroke();
            ctx.fillText(String(j + 1), x, topPad + matrixSize + 7);
          }}
          ctx.font = 'bold 12px sans-serif';
          ctx.fillText('Scored residue', leftMargin + matrixSize / 2, topPad + matrixSize + 28);
        }}

        function drawChainBoundaries() {{
          var cb = document.getElementById('boundaries_{uid}');
          if (!cb || !cb.checked) return;
          var boundaries = [];
          for (var bi = 1; bi < N; bi++) {{
            if (chainIds[bi] !== chainIds[bi - 1]) boundaries.push(bi);
          }}
          ctx.save();
          ctx.strokeStyle = 'black';
          ctx.lineWidth = 1.2;
          ctx.setLineDash([4, 3]);
          for (var k = 0; k < boundaries.length; k++) {{
            var rowPos = topPad + boundaries[k] * cellSize;
            ctx.beginPath();
            ctx.moveTo(leftMargin, rowPos);
            ctx.lineTo(leftMargin + matrixSize, rowPos);
            ctx.stroke();

            var colPos = leftMargin + boundaries[k] * cellSize;
            ctx.beginPath();
            ctx.moveTo(colPos, topPad);
            ctx.lineTo(colPos, topPad + matrixSize);
            ctx.stroke();
          }}
          ctx.restore();
        }}

        function redraw(sel) {{
          ctx.imageSmoothingEnabled = false;
          ctx.clearRect(0, 0, canvas.width, canvas.height);
          ctx.drawImage(offCanvas, 0, 0, N, N, leftMargin, topPad, matrixSize, matrixSize);
          drawAxes();
          drawChainBoundaries();
          if (sel) {{
            ctx.strokeStyle = 'red';
            ctx.lineWidth = 2;
            ctx.strokeRect(leftMargin + sel.c1 * cellSize, topPad + sel.r1 * cellSize,
                            (sel.c2 - sel.c1 + 1) * cellSize, (sel.r2 - sel.r1 + 1) * cellSize);
          }}
        }}

        function cellFromEvent(e) {{
          var rect = canvas.getBoundingClientRect();
          var x = e.clientX - rect.left - leftMargin;
          var y = e.clientY - rect.top - topPad;
          return {{row: Math.max(0, Math.min(N - 1, Math.floor(y / cellSize))),
                    col: Math.max(0, Math.min(N - 1, Math.floor(x / cellSize)))}};
        }}

        function highlight(sel) {{
          viewer.setStyle({{}}, {{cartoon: {{color: 'gray'}}}});
          var byChain = {{}};
          function addRange(a, b) {{
            for (var k = a; k <= b; k++) {{
              var ch = chainIds[k], rn = resiNums[k];
              if (!byChain[ch]) byChain[ch] = [];
              byChain[ch].push(rn);
            }}
          }}
          addRange(sel.r1, sel.r2);
          addRange(sel.c1, sel.c2);
          for (var ch in byChain) {{
            viewer.setStyle({{chain: ch, resi: byChain[ch]}}, {{cartoon: {{color: 'red'}}}});
          }}
          viewer.render();
        }}

        var mode = 'ipsae';
        var currentSel = null;
        drawBase(mode);
        redraw(null);

        var dragStart = null;
        canvas.addEventListener('mousedown', function(e) {{ dragStart = cellFromEvent(e); }});
        canvas.addEventListener('mousemove', function(e) {{
          if (!dragStart) return;
          var cur = cellFromEvent(e);
          redraw({{r1: Math.min(dragStart.row, cur.row), r2: Math.max(dragStart.row, cur.row),
                   c1: Math.min(dragStart.col, cur.col), c2: Math.max(dragStart.col, cur.col)}});
        }});
        canvas.addEventListener('mouseup', function(e) {{
          if (!dragStart) return;
          var cur = cellFromEvent(e);
          var sel = {{r1: Math.min(dragStart.row, cur.row), r2: Math.max(dragStart.row, cur.row),
                      c1: Math.min(dragStart.col, cur.col), c2: Math.max(dragStart.col, cur.col)}};
          currentSel = sel;
          redraw(sel);
          highlight(sel);
          dragStart = null;
        }});

        var vElement = document.getElementById('viewer_{uid}');
        vElement.addEventListener('dblclick', function() {{
          viewer.setStyle({{}}, baseStyle);
          viewer.render();
          currentSel = null;
          redraw(null);
        }});

        var scoreModeSelect = document.getElementById('scoremode_{uid}');
        scoreModeSelect.addEventListener('change', function() {{
          mode = scoreModeSelect.value;
          drawBase(mode);
          redraw(currentSel);
        }});

        var boundariesCheckbox = document.getElementById('boundaries_{uid}');
        boundariesCheckbox.addEventListener('change', function() {{
          redraw(currentSel);
        }});
      }});
    }})();
    </script>
    """
    return html


if not cifs:
    print("No .cif files found under ./runs yet - upload a run's output.zip first.")
else:
    # Sort models by decreasing ipTM (from each model's summary_confidence.json).
    cifs_by_iptm = sorted(cifs, key=lambda c: (get_iptm_ptm(c)[0] if get_iptm_ptm(c)[0] is not None else -1.0),
                          reverse=True)

    def _dropdown_label(c):
        iptm, _ = get_iptm_ptm(c)
        return f"{model_label(c)} (ipTM {iptm:.2f})" if iptm is not None else model_label(c)

    model_dropdown = widgets.Dropdown(
        options=[(_dropdown_label(c), c) for c in cifs_by_iptm],
        description="Model:",
    )
    viewer_output = widgets.Output()

    def _on_change(change):
        if change["name"] == "value":
            with viewer_output:
                viewer_output.clear_output(wait=True)
                display(HTML(_build_model_html(change["new"])))

    model_dropdown.observe(_on_change, names="value")
    display(model_dropdown, viewer_output)
    with viewer_output:
        display(HTML(_build_model_html(cifs_by_iptm[0])))

In [ ]:
#@title Compute ipSAE for every model and rank by decreasing ipSAE
# Uses the Dunbrack Lab's ipsae.py (https://github.com/DunbrackLab/IPSAE),
# which reads AlphaFold3-style keys ("pae", "atom_plddts") from a
# "*_full_data*.json" file and an optional sibling "*_summary_confidences*.json"
# (for "chain_pair_iptm", used to populate the ipTM_af column). OpenDDE's
# full_data.json uses different key names ("token_pair_pae", "atom_plddt") and
# a different sidecar naming convention, so we adapt/copy each sample's JSON
# into ipsae.py's expected shape before invoking it.
#
# ipSAE is an INTER-CHAIN metric: it needs >=2 distinct chains (asym_ids) to
# score an interface. If a model's input JSON only had one proteinChain entry
# (or OpenDDE's log for that pred run showed "N_asym 1"), there is no
# interface to score, and ipsae.py will only ever write its header line - it
# opens its output files before it parses anything, so a single-chain (or
# otherwise failing) run looks like an "empty" .txt/.pml rather than an
# obvious error.
import glob
import json
import os
import subprocess
import sys

import pandas as pd

!curl -fsSL -o ipsae.py https://raw.githubusercontent.com/DunbrackLab/IPSAE/main/ipsae.py

PAE_CUTOFF = 10  # @param {type:"number"}
DIST_CUTOFF = 10  # @param {type:"number"}
IPSAE_TMP_DIR = "ipsae_tmp"
os.makedirs(IPSAE_TMP_DIR, exist_ok=True)

rows = []
for cif_path in cifs:
    full_data_path = find_full_data_json(cif_path)
    if full_data_path is None:
        print(f"Skipping {cif_path}: no *_full_data_sample_*.json "
              f"(re-run pred with --need_atom_confidence true)")
        continue

    try:
        with open(full_data_path) as f:
            full_data = json.load(f)
        if "token_pair_pae" not in full_data or "atom_plddt" not in full_data:
            print(f"Skipping {cif_path}: full_data.json missing token_pair_pae/atom_plddt")
            continue

        n_chains = len(set(full_data.get("token_asym_id", [])))
        if n_chains < 2:
            print(f"Skipping {cif_path}: only {n_chains} chain(s) detected in "
                  f"token_asym_id - ipSAE needs >=2 chains. Check that the input "
                  f"JSON's \"sequences\" list had one entry per chain, not one "
                  f"combined sequence.")
            continue

        stem = model_label(cif_path).replace(":", "").replace(" ", "_").replace(",", "")

        # Adapted PAE/confidence file, named so ipsae.py's "full_data" path
        # substitution can find our adapted summary file below.
        adapted_pae_path = os.path.join(IPSAE_TMP_DIR, f"{stem}_full_data.json")
        with open(adapted_pae_path, "w") as f:
            json.dump({"pae": full_data["token_pair_pae"],
                       "atom_plddts": full_data["atom_plddt"]}, f)

        summary_path = find_summary_json(cif_path)
        if summary_path is not None:
            with open(summary_path) as f:
                summary_data = json.load(f)
            if "chain_pair_iptm" in summary_data:
                adapted_summary_path = os.path.join(IPSAE_TMP_DIR, f"{stem}_summary_confidences.json")
                with open(adapted_summary_path, "w") as f:
                    json.dump({"chain_pair_iptm": summary_data["chain_pair_iptm"]}, f)

        result = subprocess.run(
            [sys.executable, "ipsae.py", adapted_pae_path, cif_path,
             str(PAE_CUTOFF), str(DIST_CUTOFF)],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            print(f"ipsae.py failed on {cif_path} (exit {result.returncode}):")
            print("--- stdout ---\n" + result.stdout)
            print("--- stderr ---\n" + result.stderr)
            continue

        score_files = [p for p in glob.glob(cif_path.replace(".cif", "") + "*.txt")
                       if not p.endswith("_byres.txt")]
        if not score_files:
            print(f"No ipsae.py output found for {cif_path}")
            print("--- stdout ---\n" + result.stdout)
            print("--- stderr ---\n" + result.stderr)
            continue

        with open(score_files[0]) as f:
            lines = [l for l in f if l.strip()]
        if len(lines) < 2:
            # ipsae.py opens its output files before parsing anything, so a
            # header-only file usually means the run produced no chain pairs
            # (e.g. the .cif didn't parse the way ipsae.py expected) rather
            # than a clean "no interface" result.
            print(f"ipsae.py produced no data rows for {cif_path} (header only).")
            print("--- ipsae.py stdout ---\n" + result.stdout)
            print("--- ipsae.py stderr ---\n" + result.stderr)
            continue

        header = lines[0].split()
        scores = pd.DataFrame([l.split() for l in lines[1:]], columns=header)
        scores["ipSAE"] = scores["ipSAE"].astype(float)
        # Every row is already an inter-chain pair (ipsae.py skips chain1==chain2);
        # take the strongest interface as this model's representative score.
        best = scores.loc[scores["ipSAE"].idxmax()]

        rows.append({
            "model": model_label(cif_path),
            "chain_1": best["Chn1"],
            "chain_2": best["Chn2"],
            "ipSAE": float(best["ipSAE"]),
            "ipSAE_d0chn": float(best["ipSAE_d0chn"]),
            "ipTM_af": float(best["ipTM_af"]),
            "pDockQ": float(best["pDockQ"]),
        })
    except Exception as e:
        print(f"Skipping {cif_path}: {e}")

ranking_df = pd.DataFrame(rows).sort_values("ipSAE", ascending=False).reset_index(drop=True)
ranking_df.to_csv("ipsae_ranking.csv", index=False)
ranking_df

In [ ]:
from google.colab import files

files.download("ipsae_ranking.csv")